In [1]:
import os, subprocess

REPO_URL = "https://github.com/Ved29197/INDUSMIND.git"

# Colab starts empty each session, so we clone the repo and re-download the data.
if not os.path.exists("src"):
    if not os.path.exists("INDUSMIND"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("INDUSMIND")

result = subprocess.run(
    ["python", "-m", "src.data_processing.download"], capture_output=True, text=True
)
print(result.stdout, result.stderr)
print("Working directory:", os.getcwd())

Saved: /content/INDUSMIND/data/raw/ai4i2020.csv
 
Working directory: /content/INDUSMIND


In [2]:
import sys
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import plotly.express as px

from src.data_processing import schema
from src.data_processing.loader import load_ai4i

df, report = load_ai4i()
print(report.summary())

VALIDATION PASSED
  INFO:    failures_with_no_mode_flag = 9
  INFO:    mode_flag_without_failure = 18
  INFO:    n_failures = 339
  INFO:    failure_rate_pct = 3.39
  INFO:    n_rows = 10000
  INFO:    type_counts = {'L': 6000, 'M': 2997, 'H': 1003}


In [3]:
n = len(df)
n_fail = int(df[schema.TARGET].sum())
baseline_acc = 1 - n_fail / n
print(f"A 'model' that always predicts NO failure:")
print(f"  accuracy        = {baseline_acc:.2%}")
print(f"  failures caught = 0 of {n_fail}")

display(df[schema.NUMERIC_FEATURES].describe().T.round(2))

A 'model' that always predicts NO failure:
  accuracy        = 96.61%
  failures caught = 0 of 339


,count,mean,std,min,25%,50%,75%,max
Air temperature [K],10000.0,300.00,2.00,295.3,298.3,300.1,301.5,304.5
Process temperature [K],10000.0,310.01,1.48,305.7,308.8,310.1,311.1,313.8
Rotational speed [rpm],10000.0,1538.78,179.28,1168.0,1423.0,1503.0,1612.0,2886.0
Torque [Nm],10000.0,39.99,9.97,3.8,33.2,40.1,46.8,76.6
Tool wear [min],10000.0,107.95,63.65,0.0,53.0,108.0,162.0,253.0


In [4]:
modes = schema.FAILURE_MODE_COLUMNS
failed = df[schema.TARGET] == 1
any_mode = df[modes].sum(axis=1) > 0

print("Failures per cause (a failure can have more than one):")
print(df[modes].sum())

print("\n--- The 9 failures with NO cause flagged ---")
display(df[failed & ~any_mode])

print("\n--- The 18 rows with a cause flagged but NO failure: which causes? ---")
print(df[~failed & any_mode][modes].sum())

Failures per cause (a failure can have more than one):
TWF     46
HDF    115
PWF     95
OSF     98
RNF     19
dtype: int64

--- The 9 failures with NO cause flagged ---


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
1437,1438,H30851,H,298.8,309.9,1439,45.2,40,1,0,0,0,0,0
2749,2750,M17609,M,299.7,309.2,1685,28.9,179,1,0,0,0,0,0
4044,4045,M18904,M,301.9,310.9,1419,47.7,20,1,0,0,0,0,0
4684,4685,M19544,M,303.6,311.8,1421,44.8,101,1,0,0,0,0,0
5536,5537,M20396,M,302.3,311.8,1363,54.0,119,1,0,0,0,0,0
5941,5942,L53121,L,300.6,310.7,1438,48.5,78,1,0,0,0,0,0
6478,6479,L53658,L,300.5,309.8,1663,29.1,145,1,0,0,0,0,0
8506,8507,L55686,L,298.4,309.6,1710,27.3,163,1,0,0,0,0,0
9015,9016,L56195,L,297.2,308.1,1431,49.7,210,1,0,0,0,0,0



--- The 18 rows with a cause flagged but NO failure: which causes? ---
TWF     0
HDF     0
PWF     0
OSF     0
RNF    18
dtype: int64


In [5]:
tp = int((failed & any_mode).sum())
fp = int((~failed & any_mode).sum())
fn = int((failed & ~any_mode).sum())

print("If we 'predicted' failure using ONLY the cause flags:")
print(f"  precision = {tp/(tp+fp):.3f}")
print(f"  recall    = {tp/(tp+fn):.3f}")
print("A real predictor can't know the cause before the failure happens,")
print("so these columns must never be features.")

# Do the identifiers carry a signal? Failure rate per tenth of the UDI range.
decile = pd.qcut(df["UDI"], 10, labels=False)
print("\nFailure rate by UDI decile (should be roughly flat if UDI is just a row number):")
print(df.groupby(decile)[schema.TARGET].mean().round(4))

If we 'predicted' failure using ONLY the cause flags:
  precision = 0.948
  recall    = 0.973
A real predictor can't know the cause before the failure happens,
so these columns must never be features.

Failure rate by UDI decile (should be roughly flat if UDI is just a row number):
UDI
0    0.023
1    0.029
2    0.022
3    0.027
4    0.134
5    0.020
6    0.023
7    0.022
8    0.017
9    0.022
Name: Machine failure, dtype: float64


In [6]:
outcome = df[schema.TARGET].map({0: "No failure", 1: "Failure"})

for col in schema.NUMERIC_FEATURES:
    fig = px.histogram(
        df, x=col, color=outcome, barmode="overlay",
        histnorm="probability density", opacity=0.6, nbins=50,
        labels={"color": "Outcome"}, title=f"{col}: failures vs. non-failures",
    )
    fig.show()

print(df.groupby("Type")[schema.TARGET].agg(["count", "sum", "mean"]).round(4))

wear_bins = pd.cut(df["Tool wear [min]"], [0, 50, 100, 150, 200, 260], include_lowest=True)
print("\nFailure rate by tool-wear band:")
print(df.groupby(wear_bins, observed=True)[schema.TARGET].agg(["count", "mean"]).round(4))

      count  sum    mean
Type                    
H      1003   21  0.0209
L      6000  235  0.0392
M      2997   83  0.0277

Failure rate by tool-wear band:
                 count    mean
Tool wear [min]               
(-0.001, 50.0]    2396  0.0217
(50.0, 100.0]     2271  0.0233
(100.0, 150.0]    2295  0.0218
(150.0, 200.0]    2276  0.0290
(200.0, 260.0]     762  0.1549


In [7]:
corr = df[schema.NUMERIC_FEATURES + [schema.TARGET]].corr()
fig = px.imshow(
    corr.round(2), text_auto=True, color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1, title="Correlation matrix (numeric features and failure)",
)
fig.show()

In [8]:
temp_diff = df["Process temperature [K]"] - df["Air temperature [K]"]
power_w = df["Torque [Nm]"] * df["Rotational speed [rpm]"] * 2 * np.pi / 60
wear_torque = df["Tool wear [min]"] * df["Torque [Nm]"]
rpm = df["Rotational speed [rpm]"]
limit = df["Type"].map({"L": 11000, "M": 12000, "H": 13000})

# Thresholds as I recall them from the dataset documentation. VERIFY them on the
# UCI page; the output below tests them against the actual data either way.
rules = {
    "HDF": (temp_diff < 8.6) & (rpm < 1380),
    "PWF": (power_w < 3500) | (power_w > 9000),
    "OSF": wear_torque > limit,
}

print("mode | flagged & rule met | total flagged | rule met but NOT flagged")
for mode, rule in rules.items():
    flagged = df[mode] == 1
    print(mode, int((flagged & rule).sum()), int(flagged.sum()), int((~flagged & rule).sum()))

print("\nTool wear at TWF events:")
print(df.loc[df["TWF"] == 1, "Tool wear [min]"].describe().round(1))

outcome_eng = df[schema.TARGET].map({0: "No failure", 1: "Failure"})
px.histogram(
    pd.DataFrame({"power_W": power_w}), x="power_W", color=outcome_eng,
    barmode="overlay", histnorm="probability density", opacity=0.6, nbins=60,
    labels={"color": "Outcome"}, title="Mechanical power (torque × angular speed)",
).show()

mode | flagged & rule met | total flagged | rule met but NOT flagged
HDF 115 115 0
PWF 95 95 0
OSF 98 98 0

Tool wear at TWF events:
count     46.0
mean     216.4
std       12.3
min      198.0
25%      207.0
50%      214.5
75%      224.2
max      253.0
Name: Tool wear [min], dtype: float64


In [9]:
# 9a. Where exactly is the spike? Failure rate per block of 250 rows
bin250 = (df["UDI"] - 1) // 250
spike = df.groupby(bin250)[schema.TARGET].agg(["count", "sum", "mean"]).round(3)
spike.index = [f"{i*250+1}-{(i+1)*250}" for i in spike.index]
display(spike)

# 9b. Which failure causes drive it? Cause counts per UDI decile
decile = pd.qcut(df["UDI"], 10, labels=False)
display(df.groupby(decile)[schema.FAILURE_MODE_COLUMNS].sum())

# 9c. Do the sensor readings differ inside the spike region?
lo, hi = 4001, 5000   # adjust after looking at 9a
region = np.where(df["UDI"].between(lo, hi), f"UDI {lo}-{hi}", "Other rows")
display(df.groupby(region)[schema.NUMERIC_FEATURES].mean().round(2))

# 9d. Is UDI a hidden time axis? Look at air temperature along UDI
px.line(df, x="UDI", y="Air temperature [K]", title="Air temperature along UDI").show()
print("lag-1 autocorrelation of air temperature:",
      round(df["Air temperature [K]"].autocorr(lag=1), 3))

# 9e. How much of the failure label is deterministic vs. noisy?
det = df[["HDF", "PWF", "OSF"]].sum(axis=1) > 0
n_det = int((failed & det).sum())
print(f"\nFailures explained by deterministic rules (HDF/PWF/OSF): "
      f"{n_det} of {n_fail} ({n_det/n_fail:.1%})")
print(f"Failures involving only TWF, RNF or no flag: {n_fail - n_det}")
high_wear = df["Tool wear [min]"] >= 198
print(f"TWF events among rows with wear >= 198: "
      f"{int(df.loc[high_wear, 'TWF'].sum())} of {int(high_wear.sum())} rows")

,count,sum,mean
1-250,250,11,0.044
251-500,250,5,0.020
501-750,250,3,0.012
751-1000,250,4,0.016
1001-1250,250,10,0.040
1251-1500,250,9,0.036
1501-1750,250,4,0.016
1751-2000,250,6,0.024
2001-2250,250,6,0.024
2251-2500,250,6,0.024


,TWF,HDF,PWF,OSF,RNF
UDI,,,,,
0,1,0,13,10,0
1,5,0,13,12,3
2,5,0,8,8,2
3,4,7,9,9,3
4,6,108,15,12,0
5,5,0,8,9,6
6,5,0,10,8,3
7,5,0,9,9,2
8,6,0,5,7,0


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
Other rows,299.70,309.89,1537.36,40.03,107.54
UDI 4001-5000,302.72,311.09,1551.48,39.64,111.62


lag-1 autocorrelation of air temperature: 0.999

Failures explained by deterministic rules (HDF/PWF/OSF): 287 of 339 (84.7%)
Failures involving only TWF, RNF or no flag: 52
TWF events among rows with wear >= 198: 46 of 894 rows


In [10]:
blocks = spike.reset_index().rename(columns={"index": "UDI block", "mean": "failure_rate"})
px.bar(blocks, x="UDI block", y="failure_rate",
       title="Failure rate per 250-row UDI block").show()

temp_diff_by_block = (df["Process temperature [K]"] - df["Air temperature [K]"]).groupby(bin250).mean()
temp_diff_by_block.index = blocks["UDI block"]
fig = px.line(temp_diff_by_block.rename("mean_temp_diff_K").reset_index(),
              x="UDI block", y="mean_temp_diff_K",
              title="Mean (process - air) temperature per block; HDF rule needs < 8.6 K")
fig.add_hline(y=8.6, line_dash="dash")
fig.show()